In [ ]:
import pandas as pd  # Import pandas
from pandas import DataFrame  # Import DataFrame
import pymysql  # Import MySQL client
import rp  # Import rp module
import ratespricer as rp  # Import business-day utilities
import matplotlib.pyplot as plt  # Import plotting library
import numpy as np  # Import numerical library
from datetime import datetime  # Import datetime helper
import os  # Import OS helpers


# Parse a contract code, for example T2606 -> ('T', 26, 6)
def parse_contract_code(contract_code):  # Parse contract code
    prefix = ''.join(ch for ch in contract_code if ch.isalpha())  # Extract alphabetic prefix
    yy = int(contract_code[len(prefix):len(prefix) + 2])  # Extract 2-digit year
    mm = int(contract_code[-2:])  # Extract month
    return prefix, yy, mm  # Return parsed parts


# Given a quarterly contract, return the next quarterly contract
def next_quarter_contract(contract_code):  # Get next quarterly contract
    prefix, yy, mm = parse_contract_code(contract_code)  # Parse current contract
    quarter_months = [3, 6, 9, 12]  # Quarterly contract months

    if mm not in quarter_months:  # Validate contract month
        raise ValueError(f'Contract month must be one of {quarter_months}: {contract_code}')  # Raise validation error

    current_idx = quarter_months.index(mm)  # Locate current quarter index
    next_idx = (current_idx + 1) % len(quarter_months)  # Locate next quarter index
    next_yy = yy + 1 if next_idx == 0 else yy  # Advance year if needed
    next_mm = quarter_months[next_idx]  # Get next quarter month
    return f"{prefix}{next_yy:02d}{next_mm:02d}"  # Return next contract code




# Infer the nearby contract from the current date
def get_auto_near_contract(prefix, current_dt=None):  # Infer nearby contract from date
    if current_dt is None:  # Use current date by default
        current_dt = datetime.now()  # Read current datetime

    yy = current_dt.year % 100  # Use 2-digit year
    month_day = (current_dt.month, current_dt.day)  # Extract month/day tuple

    if (3, 1) <= month_day < (6, 1):  # Between Mar 1 and Jun 1
        contract_year = yy  # Set axis ranges?
        contract_month = 6  # Set axis ranges?
    elif (6, 1) <= month_day < (9, 1):  # Between Jun 1 and Sep 1
        contract_year = yy  # Set axis ranges?
        contract_month = 9  # Set axis ranges?
    elif (9, 1) <= month_day < (12, 1):  # Between Sep 1 and Dec 1
        contract_year = yy  # Set axis ranges?
        contract_month = 12  # Set axis ranges?
    else:  # Remaining date range
        # For dates before Mar 1 or on/after Dec 1, roll to next year's Mar contract.
        contract_year = yy + 1 if month_day >= (12, 1) else yy  # Roll year after Dec 1
        contract_month = 3  # Use March contract

    return f"{prefix}{contract_year:02d}{contract_month:02d}"  # Return nearby contract code

# Given a quarterly contract, return the next quarterly contractThicker line? fut_list
def generate_fut_list_from_near(start_date, start_near, current_near):  # Build full roll list
    start_prefix, start_yy, start_mm = parse_contract_code(start_near)  # Parse starting contract
    current_prefix, current_yy, current_mm = parse_contract_code(current_near)  # Parse nearby contract
    quarter_months = [3, 6, 9, 12]  # Quarterly contract months

    if start_prefix != current_prefix:  # Validate product prefix
        raise ValueError('start_near and current_near must have the same prefix')  # Raise validation error
    if start_mm not in quarter_months or current_mm not in quarter_months:  # Validate quarter months
        raise ValueError('Contract month must be one of 03/06/09/12')  # Raise validation error
    if (current_yy, current_mm) < (start_yy, start_mm):  # Check ordering
        raise ValueError('current_near cannot be earlier than start_near')  # Raise validation error

    fut_list = []  # Initialize result list
    roll_date = datetime.strptime(start_date, '%Y-%m-%d')  # Parse starting roll date
    near_contract = start_near  # Initialize nearby contract

    while True:  # Iterate until current nearby contract
        far_contract = next_quarter_contract(near_contract)  # Compute far contract
        fut_list.append((roll_date.strftime('%Y-%m-%d'), near_contract, far_contract))  # Append one roll tuple

        if near_contract == current_near:  # Stop at target nearby contract
            break  # Exit the loop

        near_contract = far_contract  # Promote far to next nearby
        next_month = roll_date.month + 3  # Advance by one quarter
        next_year = roll_date.year  # Start from current year
        if next_month > 12:  # Handle year rollover
            next_month -= 12  # Wrap month into next year
            next_year += 1  # Increment year
        roll_date = datetime(next_year, next_month, 1)  # Update next roll date

    return fut_list  # Return generated list


def process_futures_data(fut_list):  # Query and transform futures data
    """
    TODO: last trading day
    Fetch position data from T-80 to T-1 business days before expiry.
    Return all contracts for each product (TL/T/TF/TS), from TL2403 onward:
        posit      nearby-contract open interest
        posit_perc nearby roll progress = far / (near + far)
    """

    posit = DataFrame()        # Store nearby open interest
    posit_perc = DataFrame()   # Store nearby roll progress

    # Create database connection
    cnn = pymysql.connect(host='192.168.119.53',  # Set axis ranges
                          user='fangyd',  # Database user
                          passwd='fyd2025!',  # Database password
                          db='bond2',  # Database name
                          charset='utf8')  # Connection charset
    cur = cnn.cursor()  # Set axis ranges

    # Loop through each roll tuple: (expiry date, near, far)
    for ed, fut_code1, fut_code2 in fut_list:  # Iterate through roll tuples

        sql = '''  # SQL template
            SELECT DateTime, position
            FROM futures_records_1d_origin
            WHERE FutCode = '%s' AND DateTime BETWEEN '%s' and '%s'
            '''

        # Build the query window from T-80 to T-1 business days
        start = rp.d_get_bus_day(ed, -80).strftime('%Y-%m-%d')
        end   = rp.d_get_bus_day(ed, -1).strftime('%Y-%m-%d')

        # Query nearby-contract open interest
        cur.execute(sql % (fut_code1, start, end))
        rst1 = cur.fetchall()

        # Query far-contract open interest
        cur.execute(sql % (fut_code2, start, end))
        rst2 = cur.fetchall()

        # Convert query results to DataFrames
        fut1 = DataFrame.from_records(rst1, columns=['datetimes', 'position'])
        fut2 = DataFrame.from_records(rst2, columns=['datetimes', 'position'])

        # Store nearby open interest
        posit[fut_code1] = fut1['position']

        # Compute roll progress
        posit_perc[fut_code1] = fut2['position'] / (fut1['position'] + fut2['position'])

    # Reset the index to the -80..-1 business-day scale
    posit.index = posit.index - 80
    posit_perc.index = posit_perc.index - 80

    cur.close()
    cnn.close()

    return posit, posit_perc




# Plot open-interest changes and highlight the nearby contract from the mapping
def plot_combined_oi(posit_dict, contract_types, nearby_contract_map):
    """
    Plot open-interest curves in a 2x2 grid.
    posit_dict: {'TL': df, 'T': df, ...}
    contract_types: ['TL','T','TF','TS']
    """

    # Configure fonts and minus-sign rendering
    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.rcParams['axes.unicode_minus'] = False

    # Create a 2x2 subplot grid
    fig, axs = plt.subplots(2, 2, figsize=(16, 12))
    axs = axs.flatten()  # Flatten subplot array for iteration

    # Loop through each product type
    for i, contract_type in enumerate(contract_types):
        if i >= len(axs):  # Guard against too many subplots
            break

        ax = axs[i]
        posit = posit_dict[contract_type]  # Current product data
        contracts = list(posit.columns)    # Contract list
        # Use the nearby contract provided by the mapping
        Nearby_contract = nearby_contract_map.get(contract_type)
        print(f'Nearby contract: {Nearby_contract}')

        # Draw one curve per contract
        for contract in contracts:
            is_Nearby = (contract == Nearby_contract)

            # Highlight the nearby contract in red
            ax.plot(
                posit.index,
                posit[contract],
                label=contract,
                color='red' if is_Nearby else None,
                linewidth=2 if is_Nearby else 1,
                marker='o',
                markersize=5 if is_Nearby else 3
            )

            # Annotate the last valid point for the nearby contract
            if is_Nearby:
                valid = posit[contract].dropna()
                if not valid.empty:
                    ax.text(
                        valid.index[-1],
                        valid.iloc[-1],
                        f'{valid.iloc[-1]:.0f}',
                        fontsize=11,
                        fontweight='bold',
                        ha='left',
                        va='bottom'
                    )

        # Format the subplot
        ax.set_title(f'{contract_type} Open Interest Change', fontsize=13)
        ax.set_xlabel('Days to Expiry')
        ax.set_ylabel('Open Interest')
        ax.legend(loc='lower left', fontsize=9)
        ax.grid(True, linestyle='--', alpha=0.6)

        if not posit.empty:
            ax.set_xlim(posit.index.min(), posit.index.max())

    plt.tight_layout()

    # Save the figure
    today = datetime.now().strftime('%Y%m%d')
    result_folder = 'result'
    os.makedirs(result_folder, exist_ok=True)

    filename = f'four_contracts_oi_change_{today}.png'
    file_path = os.path.join(result_folder, filename)

    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    print(f'Saved image: {file_path}')

    plt.show()


# Plot roll overview with open-interest changes
# Plot the 2x2 overview: roll progress on the left axis and nearby OI changes on the right axis
def plot_roll_overview(data_dict, nearby_contract_map):
    """
    # Data format: (open interest, roll progress)
    data_dict = {
        'TL': (posit, posit_perc),
        'T':  (posit, posit_perc),
        ...
    }
    """
    
    # Configure fonts and minus-sign rendering
    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.rcParams['axes.unicode_minus'] = False

    # Create a 2x2 subplot grid
    fig, axes = plt.subplots(2, 2, figsize=(22, 16))
    axes = axes.flatten()  # Flatten subplot array for iteration

    today = datetime.now().strftime('%Y%m%d')  # Get today's date string

    # Loop through each product
    for i, (product, (posit, posit_perc)) in enumerate(data_dict.items()):

        ax = axes[i]  # Current subplot

        Nearby_contract = nearby_contract_map.get(product)
        print(f'Nearby contract: {Nearby_contract}')

        # Left axis: roll-progress curves
        for col in posit_perc.columns:

            is_Nearby = (col == Nearby_contract)

            # Use dashed lines for all contracts
            ax.plot(
                posit_perc.index,
                posit_perc[col],
                linestyle='--',              # Dashed line
                linewidth=3 if is_Nearby else 1.5,
                color='red' if is_Nearby else None,
                label=col,
                zorder=2                    # Drawing layer
            )

            # Annotate the last valid point for the nearby contract
            if is_Nearby:
                valid = posit_perc[col].dropna()
                if not valid.empty:
                    last_x = valid.index[-1]
                    last_y = valid.iloc[-1]

                    ax.scatter(last_x, last_y, s=80, color='darkred', zorder=5)  # Mark the final nearby point

                    # Keep the annotation above the bars
                    ax.annotate(
                        f"({last_x:.0f}, {last_y:.3f})",
                        (last_x, last_y),
                        xytext=(10, 10),
                        textcoords='offset points',
                        fontsize=16,
                        fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.3',
                                  facecolor='red',
                                  alpha=0.9),
                        arrowprops=dict(arrowstyle='->',
                                        color='darkred'),
                        zorder=6  # Highest drawing layer
                    )

        # Format the left axis
        ax.set_title(f"{product} Roll Progress", fontsize=14, fontweight='bold')
        ax.set_xlabel('Days to Expiry')
        ax.set_ylabel('Roll Progress')
        ax.set_xlim(-80, 0)
        ax.set_ylim(0, 1)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.legend(fontsize=8, loc='upper left')

        # Right axis: nearby-contract OI change
        ax_right = ax.twinx()  # Create right y-axis

        if Nearby_contract in posit.columns:

            oi = posit[Nearby_contract].diff().dropna()

            x_vals = oi.index
            y_vals = oi.values

            colors = ['red' if y > 0 else 'green' for y in y_vals]

            # Add black borders to the bars
            bars = ax_right.bar(
                x_vals,
                y_vals,
                color=colors,
                edgecolor='black',      # Border color
                linewidth=0.6,          # Border width
                alpha=0.7,
                width=1.0,
                label=f"{Nearby_contract} OI Change",
                zorder=1               # Draw bars below lines
            )

            ax_right.set_ylabel("Open Interest Change")
            ax_right.tick_params(axis='y')

            if len(y_vals) > 0:
                max_val = max(abs(y_vals.max()), abs(y_vals.min()))
                ax_right.set_ylim(-max_val * 1.2, max_val * 1.2)

            # Annotate the last bar
            if len(x_vals) > 0:
                last_x = x_vals[-1]
                last_y = y_vals[-1]

                bbox_color = 'lightcoral' if last_y > 0 else 'lightgreen'
                edge_color = 'darkred' if last_y > 0 else 'darkgreen'

                ax_right.annotate(
                    f'{last_y:.0f}',
                    xy=(last_x, last_y),
                    xytext=(0, 15),
                    textcoords='offset points',
                    ha='center',
                    fontsize=16,
                    fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3',
                              facecolor=bbox_color,
                              edgecolor=edge_color),
                    arrowprops=dict(arrowstyle='->',
                                    color=edge_color),
                    zorder=3   # Above bars but below line markers
                )

            ax_right.legend(loc='upper right', fontsize=8)

    # Set the overall figure title
    plt.suptitle("Bond Futures Roll Overview (with Nearby OI Change)",
                 fontsize=18,
                 fontweight='bold')

    plt.tight_layout()
    plt.subplots_adjust(top=0.93)

    # Save the figure
    today = datetime.now().strftime('%Y%m%d')
    result_folder = 'result'
    os.makedirs(result_folder, exist_ok=True)

    filename = f'bond_futures_roll_overview_with_oi_change_{today}.png'
    file_path = os.path.join(result_folder, filename)

    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    print(f'Saved image: {file_path}')

    plt.show()

# Plot roll-progress contrast for the nearby contract of each product
def plot_roll_contrast(data_dict, nearby_contract_map):
    """
    Plot the nearby-contract roll-progress contrast for TL/T/TF/TS.
    # Data format: (open interest, roll progress)
    data_dict = {
        'TL': (posit, posit_perc),
        'T':  (posit, posit_perc),
        'TF': (posit, posit_perc),
        'TS': (posit, posit_perc),
    }
    """

    # Configure fonts and minus-sign rendering
    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.rcParams['axes.unicode_minus'] = False

    fig, ax = plt.subplots(figsize=(16, 10))

    # Fixed color mapping by product
    color_map = {
        'TL': 'blue',
        'T': 'red',
        'TF': 'green',
        'TS': 'orange'
    }

    for product, (posit, posit_perc) in data_dict.items():

        if posit_perc.empty:
            continue

        # Use the nearby contract from the mapping
        Nearby_contract = nearby_contract_map.get(product)

        x = posit_perc.index
        y = posit_perc[Nearby_contract]

        ax.plot(
            x,
            y,
            linestyle='--',
            linewidth=3,
            color=color_map.get(product, None),
            label=f"{Nearby_contract}",
            zorder=2
        )

        # Annotate the last valid point
        valid = y.dropna()
        if not valid.empty:
            last_x = valid.index[-1]
            last_y = valid.iloc[-1]

            ax.scatter(
                last_x,
                last_y,
                s=120,
                color=color_map.get(product, None),
                zorder=5
            )

            ax.annotate(
                f"({last_x:.0f}, {last_y:.3f})",
                (last_x, last_y),
                xytext=(10, 10),
                textcoords='offset points',
                fontsize=14,
                fontweight='bold',
                bbox=dict(
                    boxstyle='round,pad=0.3',
                    facecolor=color_map.get(product, 'gray'),
                    alpha=0.8
                ),
                arrowprops=dict(
                    arrowstyle='->',
                    color=color_map.get(product, 'black')
                ),
                zorder=6
            )

    # Format the figure
    ax.set_title("Roll Progress Contrast", fontsize=18, fontweight='bold')
    ax.set_xlabel("Days to Expiry")
    ax.set_ylabel("Roll Progress")
    ax.set_xlim(-80, 0)
    ax.set_ylim(0, 1)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend(fontsize=12)

    plt.tight_layout()

    # Save the figure
    today = datetime.now().strftime('%Y%m%d')
    result_folder = 'result'
    os.makedirs(result_folder, exist_ok=True)

    filename = f'four_contracts_roll_progress_contrast_{today}.png'
    file_path = os.path.join(result_folder, filename)

    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    print(f'Saved image: {file_path}')

    plt.show()

# Plot roll-progress contrast for the nearby contract of each product
def plot_roll_contrast(data_dict, nearby_contract_map):
    """
    Roll-progress contrast for the nearby contract of each product.
    Adaptive y-axis:
        - cap at 1.0 when the maximum value is above 0.9
        - otherwise round up by 0.1 and add a 0.1 buffer
    """

    # Basic figure settings
    plt.rcParams['font.sans-serif'] = ['SimHei']      # Configure font rendering
    plt.rcParams['axes.unicode_minus'] = False        # Configure minus-sign rendering

    fig, ax = plt.subplots(figsize=(16, 10))          # Create figure

    # Fixed color mapping by product
    color_map = {
        'TL': 'blue',
        'T': 'red',
        'TF': 'green',
        'TS': 'orange'
    }

    # Annotation offsets to reduce overlap
    offset_map = {
        'TL': (15, 20),
        'T':  (15, -30),
        'TF': (-80, 20),
        'TS': (-80, -30)
    }

    global_max_y = 0   # Track the maximum roll-progress value

    # Loop through each product
    for product, (posit, posit_perc) in data_dict.items():

        if posit_perc.empty:      # Skip empty data
            continue

        Nearby_contract = nearby_contract_map.get(product)

        x = posit_perc.index                      # X axis: days to expiry
        y = posit_perc[Nearby_contract]           # Y axis: nearby roll progress

        # Update the global maximum
        local_max = y.max()
        if pd.notna(local_max):
            global_max_y = max(global_max_y, local_max)

        # Draw the roll-progress curve
        ax.plot(
            x,
            y,
            linestyle='--',                        # Dashed line
            linewidth=3,                           # Thicker line
            color=color_map.get(product, None),    # Fixed product color
            label=Nearby_contract,                 # Legend label
            zorder=2                               # Drawing layer
        )

        # Annotate the last valid point
        valid = y.dropna()                         # Drop missing values
        if not valid.empty:

            last_x = valid.index[-1]               # Last x value
            last_y = valid.iloc[-1]                # Last y value

            # Draw the final marker
            ax.scatter(
                last_x,
                last_y,
                s=120,
                color=color_map.get(product, None),
                zorder=5
            )

            # Get annotation offset
            dx, dy = offset_map.get(product, (10, 10))

            # Add annotation with arrow
            ax.annotate(
                f"({last_x:.0f}, {last_y:.3f})",
                (last_x, last_y),
                xytext=(dx, dy),
                textcoords='offset points',
                fontsize=14,
                fontweight='bold',
                bbox=dict(
                    boxstyle='round,pad=0.3',
                    facecolor=color_map.get(product, 'gray'),
                    alpha=0.85
                ),
                arrowprops=dict(
                    arrowstyle='->',
                    color=color_map.get(product, 'black')
                ),
                zorder=6
            )

    # Adaptive y-axis logic

    if global_max_y > 0.9:
        upper = 1.0     # Cap at 1.0 near full roll
    elif global_max_y == 0:
        upper = 0.1     # Minimum fallback scale
    else:
        # Round up to the nearest 0.1
        upper = np.ceil(global_max_y / 0.1) * 0.1
        upper += 0.1    # Add extra buffer

    # Set axis ranges
    ax.set_ylim(0, upper)
    ax.set_xlim(-80, 0)

    # Final figure formatting
    ax.set_title("Roll Progress Contrast", fontsize=18, fontweight='bold')
    ax.set_xlabel("Days to Nearby Contract Expiry")
    ax.set_ylabel("Roll Progress")
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend(fontsize=12)

    plt.tight_layout()

    # Save the figure
    today = datetime.now().strftime('%Y%m%d')
    result_folder = 'result'
    os.makedirs(result_folder, exist_ok=True)

    filename = f'four_contracts_roll_progress_contrast_{today}.png'
    file_path = os.path.join(result_folder, filename)

    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    print(f'Saved image: {file_path}')

    plt.show()

    
if __name__ == "__main__":
    # Infer the nearby contract from the current date
    fut_TL_near = get_auto_near_contract('TL')
    fut_T_near = get_auto_near_contract('T')
    fut_TF_near = get_auto_near_contract('TF')
    fut_TS_near = get_auto_near_contract('TS')

    # Generate each product's roll list from the fixed 2024-03-01 / 2403 starting point
    fut_TL_list = generate_fut_list_from_near('2024-03-01', 'TL2403', fut_TL_near)
    fut_T_list = generate_fut_list_from_near('2024-03-01', 'T2403', fut_T_near)
    fut_TF_list = generate_fut_list_from_near('2024-03-01', 'TF2403', fut_TF_near)
    fut_TS_list = generate_fut_list_from_near('2024-03-01', 'TS2403', fut_TS_near)

    # Use these nearby contracts directly when plotting
    nearby_contract_map = {
        'TL': fut_TL_near,
        'T': fut_T_near,
        'TF': fut_TF_near,
        'TS': fut_TS_near,
    }

    posit_TL, posit_perc_TL = process_futures_data(fut_TL_list)
    posit_T, posit_perc_T = process_futures_data(fut_T_list)
    posit_TF, posit_perc_TF = process_futures_data(fut_TF_list)
    posit_TS, posit_perc_TS = process_futures_data(fut_TS_list)

    # Open-interest data for the OI change chart
    posit_dict = {
        'TL': posit_TL,
        'T': posit_T,
        'TF': posit_TF,
        'TS': posit_TS,
    }

    # Data format: (open interest, roll progress)
    data_dict = {
        'TL': (posit_TL, posit_perc_TL),
        'T': (posit_T, posit_perc_T),
        'TF': (posit_TF, posit_perc_TF),
        'TS': (posit_TS, posit_perc_TS),
    }

    plot_combined_oi(posit_dict, ['TL', 'T', 'TF', 'TS'], nearby_contract_map)
    plot_roll_overview(data_dict, nearby_contract_map)
    plot_roll_contrast(data_dict, nearby_contract_map)
